In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import colors


def load_results(csv_file):
    """
    Load earthquake catalog and log-likelihood results from a CSV file.

    Parameters
    ----------
    csv_file : str
        Path to the CSV file.

    Returns
    -------
    dict
        Dictionary containing earthquake information and log-likelihood values.
    """

    data = pd.read_csv(csv_file)

    mask = data["step"].isin(["train", "val", "test"])

    return {
        "intlam": data.loc[mask, "intlam"].values,
        "ll_t": data.loc[mask, "logli_t"].values,
        "ll_s": data.loc[mask, "logli_s"].values,
        "time": data.loc[mask, "time_days"].values,
        "magnitude": data.loc[mask, "magnitude"].values,
        "longitude": data.loc[mask, "longitude"].values,
        "latitude": data.loc[mask, "latitude"].values,
        "train_num": np.sum(data["step"] == "train"),
        "val_num": np.sum(data["step"] == "val"),
        "test_num": np.sum(data["step"] == "test"),
    }


def plot_scatter(
    x_arr,
    y_arr,
    c_arr,
    training_num=None,
    validation_num=None,
    label=None,
    y_label=None,
):
    """
    Plot a scatter figure colored by the log-likelihood difference.

    Parameters
    ----------
    x_arr : ndarray
        X-axis values.
    y_arr : ndarray
        Y-axis values.
    c_arr : ndarray
        Values used for color mapping.
    training_num : int, optional
        Number of training samples.
    validation_num : int, optional
        Number of validation samples.
    label : str
        Figure label.
    y_label : str
        Y-axis label.
    """

    cmap = plt.get_cmap("coolwarm")

    vmax = np.percentile(np.abs(c_arr), 90)
    norm = colors.Normalize(vmin=-vmax, vmax=vmax)

    fig, ax = plt.subplots(figsize=(12, 5), dpi=300)

    scatter = ax.scatter(
        x_arr,
        y_arr,
        c=c_arr,
        cmap=cmap,
        norm=norm,
        s=1,
    )

    # ------------------------------------------------------------------
    # Mark train / validation / test partitions
    # ------------------------------------------------------------------

    if training_num is not None and validation_num is not None:

        ax.axvline(training_num, color="grey", linestyle="--")
        ax.axvline(training_num + validation_num, color="grey", linestyle="--")

        text_props = dict(
            fontsize=10,
            fontfamily="serif",
            bbox=dict(
                facecolor="white",
                edgecolor="black",
                boxstyle="square,pad=0.3",
                alpha=0.9,
            ),
            ha="center",
            va="top",
        )

        ylim = ax.get_ylim()
        xlim = ax.get_xlim()

        ax.text(
            training_num / 2,
            ylim[1] * 0.95,
            "Training Set",
            **text_props,
        )

        ax.text(
            training_num + validation_num / 2,
            ylim[1] * 0.95,
            "Validation Set",
            **text_props,
        )

        ax.text(
            training_num
            + validation_num
            + (xlim[1] - (training_num + validation_num)) / 2,
            ylim[1] * 0.95,
            "Test Set",
            **text_props,
        )

    # ------------------------------------------------------------------
    # Axis formatting
    # ------------------------------------------------------------------

    ax.set_xlabel("Index of Earthquakes", fontsize=14)
    ax.set_ylabel(y_label, fontsize=14)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["bottom"].set_linewidth(1.5)
    ax.spines["left"].set_linewidth(1.5)

    ax.tick_params(width=1.5, labelsize=12)

    ax.grid(True, linestyle="--", alpha=0.5)

    # ------------------------------------------------------------------
    # Colorbar
    # ------------------------------------------------------------------

    cbar = fig.colorbar(
        scatter,
        ax=ax,
        shrink=0.9,
        orientation="vertical",
        extend="both",
        pad=0.03,
        aspect=20,
    )

    if "spatial" in label:
        cbar.set_label(
            r"$\mathcal{L}_{Spatial,\ ST\!-\!NKF}"
            r"-"
            r"\mathcal{L}_{Spatial,\ ETAS}$",
            fontsize=13,
        )

    elif "temporal" in label:
        cbar.set_label(
            r"$\mathcal{L}_{Temporal,\ ST\!-\!NKF}"
            r"-"
            r"\mathcal{L}_{Temporal,\ ETAS}$",
            fontsize=13,
        )

    cbar.ax.tick_params(labelsize=11)

    fig.tight_layout()

    fig.savefig(
        f"./fig/IG-{label}-{y_label}.png",
        dpi=500,
        bbox_inches="tight",
    )

    plt.close(fig)




In [2]:
# ======================================================================
# Main
# ======================================================================

datasets = [
    "ComCat",
    "SaltonSea",
    "SanJac",
    "WHITE",
    "SCEDC_20",
    "SCEDC_25",
    "SCEDC_30",
]

for dataset in datasets:

    # Load ETAS results
    etas = load_results(
        f"./csv/{dataset}_empirical_empirical_empirical.csv"
    )

    # Load ST-NKF results
    nkf = load_results(
        f"./csv/{dataset}_neural_neural_neural.csv"
    )

    print(
        f"{dataset}: "
        f"train={etas['train_num']}, "
        f"val={etas['val_num']}, "
        f"test={etas['test_num']}"
    )

    # ------------------------------------------------------------------
    # Spatial log-likelihood difference
    # ------------------------------------------------------------------

    plot_scatter(
        x_arr=np.arange(1, len(etas["magnitude"]) + 1),
        y_arr=etas["magnitude"],
        c_arr=nkf["ll_s"] - etas["ll_s"],
        training_num=etas["train_num"],
        validation_num=etas["val_num"],
        label=f"{dataset}_spatial",
        y_label="Magnitude",
    )


ComCat: train=40701, val=14741, test=21888
SaltonSea: train=36639, val=2683, test=4104
SanJac: train=11770, val=3449, test=4400
WHITE: train=18548, val=13812, test=24080
SCEDC_20: train=81433, val=23484, test=13071
SCEDC_25: train=25257, val=9089, test=5000
SCEDC_30: train=6815, val=3135, test=1898
